# Integración de Claude con Herramientas de Negocio

> Aprende a integrar Claude con las herramientas SaaS más usadas en empresas:
> Notion, Google Workspace, Slack, Jira y CRMs.

## Objetivos
- Integrar Claude con Google Docs/Sheets vía API
- Construir un bot de Slack con respuestas inteligentes
- Sincronizar Claude con Notion para gestión de conocimiento
- Automatizar actualizaciones de tickets Jira/Linear

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic requests pandas --quiet

## 2. Patrones de integración con herramientas SaaS

In [ ]:
import anthropic
import json
import pandas as pd
from datetime import datetime

client = anthropic.Anthropic()
MODELO = "claude-haiku-4-5-20251001"

print("""=== PATRONES DE INTEGRACIÓN CON HERRAMIENTAS DE NEGOCIO ===

Existen 3 patrones principales para integrar LLMs con herramientas SaaS:

1. PULL (IA consulta la herramienta):
   → Claude llama a la API de la herramienta para leer datos
   → Ejemplo: leer tickets de Jira para generar resumen
   → Casos: reporting, análisis, síntesis

2. PUSH (IA actualiza la herramienta):
   → Claude procesa datos y escribe en la herramienta
   → Ejemplo: crear tareas en Notion desde emails
   → Casos: automatización de tareas, documentación

3. BIDIRECCIONAL (IA como middleware):
   → Herramienta A → Claude procesa → Herramienta B
   → Ejemplo: email cliente → Claude → ticket Jira + respuesta Gmail
   → Casos: pipelines complejos de negocio

APIs clave:
  Google Workspace: googleapis (Sheets, Docs, Drive, Gmail)
  Slack: slack-sdk
  Notion: notion-client
  Jira: atlassian-python-api
  HubSpot/Salesforce: APIs REST propias
""")

## 3. Integración con Google Sheets (simulada)

In [ ]:
# Simular datos de Google Sheets (en producción: usar gspread o googleapis)
DATOS_SHEETS = {
    "ventas_mensuales": [
        {"mes": "Enero", "ventas": 45200, "objetivo": 40000, "equipo": "Norte"},
        {"mes": "Febrero", "ventas": 38100, "objetivo": 40000, "equipo": "Norte"},
        {"mes": "Marzo", "ventas": 52300, "objetivo": 45000, "equipo": "Norte"},
        {"mes": "Enero", "ventas": 31200, "objetivo": 35000, "equipo": "Sur"},
        {"mes": "Febrero", "ventas": 36800, "objetivo": 35000, "equipo": "Sur"},
        {"mes": "Marzo", "ventas": 41500, "objetivo": 40000, "equipo": "Sur"},
    ]
}

def analizar_sheets_con_ia(datos: list, objetivo_analisis: str) -> str:
    """Analiza datos de Sheets con Claude y genera informe."""
    df = pd.DataFrame(datos)
    resumen = df.to_string(index=False)

    prompt = f"""Analiza estos datos de ventas y genera un informe ejecutivo.

Objetivo: {objetivo_analisis}

Datos:
{resumen}

El informe debe incluir (en español, máx 200 palabras):
- Tendencia general
- Mejor y peor rendimiento
- Cumplimiento de objetivos
- Recomendación accionable"""

    return client.messages.create(
        model=MODELO, max_tokens=400,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text

informe = analizar_sheets_con_ia(
    DATOS_SHEETS["ventas_mensuales"],
    "Evaluar rendimiento comercial Q1 2024 por equipo"
)
print("=== INFORME GENERADO DESDE GOOGLE SHEETS ===")
print(informe)

# En producción con gspread:
print("""
--- Código de producción con gspread ---
import gspread
from oauth2client.service_account import ServiceAccountCredentials

scope = ['https://spreadsheets.google.com/feeds']
creds = ServiceAccountCredentials.from_json_keyfile_name('credentials.json', scope)
gc = gspread.authorize(creds)
sheet = gc.open('Ventas 2024').sheet1
datos = sheet.get_all_records()
informe = analizar_sheets_con_ia(datos, 'Análisis mensual')
# Escribir informe en nueva hoja
sheet2 = gc.open('Ventas 2024').worksheet('Informes')
sheet2.update('A1', [[informe]])
""")

## 4. Bot de Slack con respuestas inteligentes

In [ ]:
# Base de conocimiento del equipo
BASE_CONOCIMIENTO = """
Políticas de la empresa:
- Horario: lunes-viernes 9:00-18:00
- Reuniones: mínimo 24h de aviso
- Vacaciones: solicitar en RRHH con 15 días de antelación
- VPN: obligatoria para acceso remoto (IT te da credenciales)
- Herramientas: Slack, Notion, Google Workspace, Jira
- Gastos: aprobar con manager antes de >100€, formulario en Notion
"""

# Simulación de mensajes de Slack al bot @IA-Asistente
MENSAJES_SLACK = [
    {"usuario": "carlos.ruiz", "canal": "#general",
     "mensaje": "@IA-Asistente ¿cuántos días de preaviso necesito para pedir vacaciones?"},
    {"usuario": "ana.martinez", "canal": "#it-soporte",
     "mensaje": "@IA-Asistente no consigo conectarme desde casa, me pide VPN pero no sé cómo"},
    {"usuario": "pedro.lopez", "canal": "#comercial",
     "mensaje": "@IA-Asistente necesito aprobar un gasto de 350€ para una cena con cliente"},
    {"usuario": "lucia.garcia", "canal": "#general",
     "mensaje": "@IA-Asistente resume los cambios del sprint 23 de Jira"},
]

def responder_slack(mensaje: dict) -> dict:
    """Genera respuesta del bot de Slack."""
    prompt = f"""Eres el asistente IA del equipo en Slack. Responde de forma directa y útil.

Base de conocimiento:
{BASE_CONOCIMIENTO}

Canal: {mensaje['canal']}
Usuario: @{mensaje['usuario']}
Pregunta: {mensaje['mensaje']}

Responde:
1. En tono informal pero profesional (estilo Slack)
2. Máximo 3 líneas
3. Si no sabes la respuesta con certeza, di a quién preguntar
4. Usa emojis con moderación"""

    respuesta = client.messages.create(
        model=MODELO, max_tokens=150,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text
    return {**mensaje, "respuesta_bot": respuesta}

print("=== BOT DE SLACK ===")
for msg in MENSAJES_SLACK:
    resultado = responder_slack(msg)
    print(f"\n[{resultado['canal']}] @{resultado['usuario']}: {resultado['mensaje'][:60]}")
    print(f"   🤖 @IA-Asistente: {resultado['respuesta_bot'].strip()[:120]}")

## 5. Integración con Notion y Jira (templates de código)

In [ ]:
# Templates de integración para producción

NOTION_INTEGRATION = '''
# Integración con Notion API
import anthropic
import requests

NOTION_TOKEN = "secret_XXXX"  # Desde Notion Integrations
DATABASE_ID = "XXXX"  # ID de la base de datos Notion
client = anthropic.Anthropic()

def crear_pagina_notion(titulo: str, contenido: str, categoria: str):
    """Crea una página en Notion con contenido generado por IA."""
    headers = {"Authorization": f"Bearer {NOTION_TOKEN}",
               "Notion-Version": "2022-06-28",
               "Content-Type": "application/json"}
    payload = {
        "parent": {"database_id": DATABASE_ID},
        "properties": {
            "Título": {"title": [{"text": {"content": titulo}}]},
            "Categoría": {"select": {"name": categoria}},
            "Fecha": {"date": {"start": datetime.now().isoformat()}}
        },
        "children": [{"object": "block", "type": "paragraph",
                      "paragraph": {"rich_text": [{"text": {"content": contenido}}]}}]
    }
    return requests.post("https://api.notion.com/v1/pages", headers=headers, json=payload)

# Uso: email → resumen IA → crear página Notion
resumen = client.messages.create(model="claude-haiku-4-5-20251001", max_tokens=300,
    messages=[{"role": "user", "content": f"Resume este email en notas de reunión: {email_texto}"}]
).content[0].text
crear_pagina_notion("Notas reunión " + datetime.now().strftime("%d/%m"), resumen, "Reunión")
'''

JIRA_INTEGRATION = '''
# Integración con Jira REST API
import anthropic
import requests
from base64 import b64encode

JIRA_URL = "https://tuempresa.atlassian.net"
JIRA_EMAIL = "tu@email.com"
JIRA_TOKEN = "ATATT_XXXX"  # API Token de Atlassian
auth = b64encode(f"{JIRA_EMAIL}:{JIRA_TOKEN}".encode()).decode()
headers = {"Authorization": f"Basic {auth}", "Content-Type": "application/json"}
client = anthropic.Anthropic()

def crear_ticket_jira(resumen_ia: str, descripcion: str, proyecto: str = "SUP"):
    """Crea ticket Jira con summary y descripción generados por IA."""
    summary = client.messages.create(model="claude-haiku-4-5-20251001", max_tokens=50,
        messages=[{"role": "user", "content": f"En máx 60 chars, título de ticket Jira para: {resumen_ia}"}]
    ).content[0].text.strip()
    payload = {"fields": {"project": {"key": proyecto}, "summary": summary,
               "description": descripcion, "issuetype": {"name": "Bug"}}}
    return requests.post(f"{JIRA_URL}/rest/api/3/issue", headers=headers, json=payload)
'''

# Guardar templates
with open("integracion_notion.py", "w", encoding="utf-8") as f:
    f.write(NOTION_INTEGRATION)
with open("integracion_jira.py", "w", encoding="utf-8") as f:
    f.write(JIRA_INTEGRATION)

print("✓ Templates de integración guardados:")
print("  - integracion_notion.py")
print("  - integracion_jira.py")
print("""
=== RESUMEN DE INTEGRACIONES ===

Herramienta  | Librería Python       | Auth           | Documentación
-------------|----------------------|----------------|---------------------------
Google Sheets| gspread              | Service Account| gspread.readthedocs.io
Gmail        | google-api-python    | OAuth 2.0      | developers.google.com
Slack        | slack-sdk            | Bot Token      | api.slack.com
Notion       | requests (REST)      | Integration Key| developers.notion.com
Jira         | atlassian-python-api | API Token      | developer.atlassian.com
HubSpot      | hubspot-api-client   | API Key        | developers.hubspot.com
Airtable     | pyairtable           | API Key        | airtable.com/developers

Consejo: usa variables de entorno para todas las credenciales (python-dotenv)
""")